# feeding / exchanger 진단 모델 v2 + 사용 가능성 검증

v1 에서 두 가지를 고쳤다.

**① feeding — 유출 보정.** 수위 변화는 `유입 − 유출` 이라 순유량이지 밸브 유량이
아니다. feeding 직전 120초 수위 기울기로 유출률을 재서 빼낸다. 실측 유출률이
P5 는 분당 −1.98, I3 는 −1.07 로 작지 않다. 분산 개선은 P5 만 21% 로 크고 나머지는
1~3% 인데(유출률이 꽤 일정해서 거의 상수 오프셋처럼 작용), **생산량이 바뀌면
유출률도 바뀌므로 보정 없는 지표는 밸브가 멀쩡해도 드리프트해 오경보를 낸다.**
정확도보다 견고성 때문에 넣는다.

**② exchanger — 전후 기울기 차이(Δ기울기).** "신호가 간 뒤 온도가 꺾였는가" 를
직접 본다. 직전 기울기와 직후 기울기의 차이라서 **"온도가 높을 때 켜진다"는 선택
편향이 자동으로 상쇄**된다. 조건을 회귀로 빼낼 필요가 없다.

## 창 길이는 탱크의 진동 주기에 맞춰 정한다

v1 에서 P5 만 부호가 반대(+11.9)로 나왔는데, **원인은 창 길이였다.** P5 의 온도
진동 주기는 237초(4분)인데 창이 120+60=180초로 **주기의 76%** 였다. 앞뒤 창이
톱니파의 서로 다른 위상에 놓여서 진동이 exchanger 효과를 덮은 것이다. 창을 줄이면
부호가 뒤집힌다:

| P5 창 | 120/60 | 60/30 | 30/15 | 20/10 |
|---|---|---|---|---|
| 차이 | +11.79 | +7.87 | +3.24 | **−0.20** |

주기는 탱크마다 4분~521분으로 100배 넘게 차이 난다. 그래서 **창을 주기의 일정
비율 이하로 제한**하고, 그 조건에서 기울기를 잴 만큼 길이가 안 나오면
**측정 불가로 표시**한다.

## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from Tank_Temp_Factor_Analysis import TANKS, WAGON_TAG

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

SAVE_DIR = "extracted_csv"
rng = np.random.default_rng(0)

EXCLUDE_HOURS = (0, 7)
RUNNING_WINDOW_MIN = 30
MIN_EVENT_SEC = 5
TAIL_SEC = 60
OUTFLOW_PRE_SEC = 120        # 유출률을 재는 구간 (feeding 직전)

BASELINE_WINDOW = 30
Z_CLIP = 4.0
CUSUM_K = 0.5
THRESH_MARGIN = 1.3
OCCURRED_FRAC = 0.25

#: exchanger 창은 주기의 이 비율을 넘지 않는다. 넘으면 톱니파가 효과를 덮는다.
WINDOW_FRAC = 0.15
#: 그래도 기울기를 재려면 최소 이만큼은 있어야 한다. 못 맞추면 측정 불가.
MIN_PRE_SEC = 20
MAX_PRE_SEC = 120

## 1. 로드 · 진동 주기 측정 · 창 길이 결정

In [ ]:
_CACHE, WINDOWS = {}, {}


def load(tank_key):
    if tank_key in _CACHE:
        return _CACHE[tank_key]
    cfg = TANKS[tank_key]
    path = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tank_key}_Temp_vs_Factors_*.csv")))[-1]
    use = ["Time", "source", WAGON_TAG, cfg["temp"], "level_f",
           cfg["exchanger"], cfg["feeding"], cfg["heat"]]
    raw = pd.read_csv(path, index_col="Time", parse_dates=True, usecols=use).sort_index()
    d = pd.DataFrame({
        "T": raw[cfg["temp"]], "level": raw["level_f"],
        "exch": raw[cfg["exchanger"]], "feed": raw[cfg["feeding"]],
        "heat": raw[cfg["heat"]], "wagon": raw[WAGON_TAG],
        "diag": (raw["source"] == "DIAGNOSTIC").astype(float)})
    w = d["wagon"].resample("1min").last().ffill()
    run = (w.diff().abs() > 0).rolling(RUNNING_WINDOW_MIN, min_periods=1).max()
    d["running"] = run.reindex(d.index, method="ffill").fillna(0)
    h = d.index.hour
    d["ok"] = (((h < EXCLUDE_HOURS[0]) | (h >= EXCLUDE_HOURS[1]))
               & (d["diag"] == 0) & (d["running"] == 1))
    _CACHE[tank_key] = d
    return d


def events(d, col):
    s = (d[col] == 1) & d["ok"]
    grp = (s & ~s.shift(1, fill_value=False)).cumsum()[s]
    return [(i.iloc[0], i.iloc[-1]) for _, i in d.index[s].to_series().groupby(grp)
            if (i.iloc[-1] - i.iloc[0]).total_seconds() >= MIN_EVENT_SEC]


def cycle_period(d):
    """heat 이 꺼졌다 켜지는 간격 = 온도 톱니파의 주기."""
    s = d["heat"][d["ok"]]
    on = s.index[(s == 1) & (s.shift(1) == 0)]
    if len(on) < 5:
        return np.nan
    return float(pd.Series(on).diff().dt.total_seconds().median())


rows = []
for tk in TANKS:
    d = load(tk)
    per = cycle_period(d)
    if np.isnan(per):
        pre, post, note = MAX_PRE_SEC, MAX_PRE_SEC // 2, "주기 측정불가(heat 거의 고정) — 오염 위험 낮음"
    else:
        total = per * WINDOW_FRAC
        pre = int(np.clip(total * 2 / 3, 0, MAX_PRE_SEC))
        post = int(pre / 2)
        note = "OK" if pre >= MIN_PRE_SEC else f"창이 {pre}초로 너무 짧음 → 측정불가"
    WINDOWS[tk] = (pre, post, note)
    rows.append({"탱크": tk, "주기(분)": round(per / 60, 1) if per == per else None,
                 "전창(초)": pre, "후창(초)": post,
                 "창/주기": f"{(pre + post) / per * 100:.0f}%" if per == per else "-",
                 "비고": note})
pd.DataFrame(rows)

## 2. 공통 통계 도구

In [ ]:
def _mad(base):
    med = np.median(base)
    s = 1.4826 * np.median(np.abs(base - med))
    if s < 1e-9:
        s = np.std(base) if np.std(base) > 1e-9 else 1.0
    return med, s


def rolling_z(v, window=BASELINE_WINDOW, clip=Z_CLIP):
    z = np.full(len(v), np.nan)
    for i in range(window, len(v)):
        med, s = _mad(v[i - window:i])
        z[i] = (v[i] - med) / s
    return np.clip(z, -clip, clip)


def cusum_down(z, k=CUSUM_K):
    s, out = 0.0, []
    for x in z:
        s = 0.0 if np.isnan(x) else max(0.0, s - x - k)
        out.append(s)
    return np.array(out)


def slope(s):
    if len(s) < 3:
        return np.nan
    x = (s.index - s.index[0]).total_seconds().values
    return np.polyfit(x, s.values, 1)[0] * 60 if x[-1] > 0 else np.nan


def calibrate(v):
    """정상 데이터에서 관측된 CUSUM 최대값으로 임계값을 정한다 (교과서 5 는 오경보 폭발)."""
    sign = np.sign(np.median(v)) or 1.0
    cs = cusum_down(rolling_z(v * sign))
    return max(float(cs.max()) * THRESH_MARGIN, 5.0), sign, cs

## 3. feeding 모델 (유출 보정)

```
유출률   = feeding 직전 120초 수위 기울기            (음수)
순유량   = (수위 끝+60s − 수위 시작−5s) / 지속시간
밸브유량 = 순유량 − 유출률          ← 이게 실제 밸브가 낸 유량
```

In [ ]:
def feeding_table(tank_key):
    d = load(tank_key)
    lv = d["level"]
    noise = lv.diff().abs().median() * 1.4826 + 1e-9

    rows = []
    for t0, t1 in events(d, "feed"):
        pre_seg = lv.loc[t0 - pd.Timedelta(seconds=OUTFLOW_PRE_SEC): t0].dropna()
        out_rate = slope(pre_seg)                       # 분당, 음수
        a = lv.asof(t0 - pd.Timedelta(seconds=5))
        b = lv.asof(t1 + pd.Timedelta(seconds=TAIL_SEC))
        dur = (t1 - t0).total_seconds()
        if np.isnan(out_rate) or pd.isna(a) or pd.isna(b) or dur <= 0:
            continue
        net = (b - a) / dur * 60
        seg = lv.loc[t0: t1 + pd.Timedelta(seconds=TAIL_SEC)]
        over = seg[seg > a + 3 * noise]
        rows.append({"start": t0, "지속초": round(dur, 1), "유입량": round(b - a, 3),
                     "유출률": round(out_rate, 3), "순유량": round(net, 3),
                     "밸브유량": round(net - out_rate, 3),
                     "지연초": round((over.index[0] - t0).total_seconds(), 1) if len(over) else np.nan})
    return pd.DataFrame(rows)


def feeding_model(tank_key):
    e = feeding_table(tank_key)
    if len(e) < BASELINE_WINDOW + 20:
        return e, {"탱크": tank_key, "이벤트": len(e), "상태": "표본부족"}

    e["발생"] = e["유입량"] >= e["유입량"].median() * OCCURRED_FRAC
    thr, sign, cs = calibrate(e["밸브유량"].values.astype(float))
    e["z"] = rolling_z(e["밸브유량"].values.astype(float) * sign)
    e["cusum"] = cs
    info = {"탱크": tank_key, "이벤트": len(e), "상태": "정상",
            "유출률중앙": round(e["유출률"].median(), 3),
            "밸브유량중앙": round(e["밸브유량"].median(), 3),
            "지연중앙(초)": round(e["지연초"].median(), 1),
            "미발생": int((~e["발생"]).sum()),
            "임계값": round(thr, 1), "현재CUSUM": round(float(cs[-1]), 2)}
    return e, info


FEED, rows = {}, []
for tk in TANKS:
    FEED[tk], info = feeding_model(tk)
    rows.append(info)
pd.DataFrame(rows)

## 4. exchanger 모델 (Δ기울기, 주기 기반 창)

```
Δ기울기 = (신호 직후 post초 온도기울기) − (신호 직전 pre초 온도기울기)
```
정상이면 음수(온도가 꺾여 내려감)여야 한다. 대조군은 **exchanger 가 꺼져 있고
최근 5분간 켜진 적 없는 무작위 시점**이다. 둘의 차이가 exchanger 의 실제 효과다.

In [ ]:
def dslope_at(d, t0, pre, post):
    a = slope(d.loc[t0 - pd.Timedelta(seconds=pre): t0, "T"].dropna())
    b = slope(d.loc[t0: t0 + pd.Timedelta(seconds=post), "T"].dropna())
    return b - a if not (np.isnan(a) or np.isnan(b)) else np.nan


def exch_model(tank_key, n_ctl=600):
    d = load(tank_key)
    pre, post, note = WINDOWS[tank_key]
    ev = events(d, "exch")

    rows = [{"start": t0, "Δ기울기": dslope_at(d, t0, pre, post)} for t0, _ in ev]
    e = pd.DataFrame(rows).dropna().reset_index(drop=True)

    # 대조군 — exchanger 가 안 켜진 시점
    idx = d.index[d["ok"] & (d["exch"] == 0)]
    ctl = []
    for i in rng.choice(len(idx), size=min(3000, len(idx)), replace=False):
        t = idx[i]
        if d.loc[t - pd.Timedelta(seconds=300): t, "exch"].max() > 0:
            continue
        v = dslope_at(d, t, pre, post)
        if not np.isnan(v):
            ctl.append(v)
        if len(ctl) >= n_ctl:
            break
    ctl = np.array(ctl)

    if len(e) < BASELINE_WINDOW + 20 or len(ctl) < 50:
        return e, ctl, {"탱크": tank_key, "이벤트": len(e), "상태": "표본부족"}

    pooled = np.sqrt((e["Δ기울기"].var() * (len(e) - 1) + ctl.var() * (len(ctl) - 1))
                     / (len(e) + len(ctl) - 2))
    effect = e["Δ기울기"].mean() - ctl.mean()
    d_coh = effect / pooled

    thr, sign, cs = calibrate(e["Δ기울기"].values.astype(float))
    e["z"] = rolling_z(e["Δ기울기"].values.astype(float) * sign)
    e["cusum"] = cs

    state = "정상" if note == "OK" or note.startswith("주기") else "측정불가"
    return e, ctl, {"탱크": tank_key, "이벤트": len(e), "상태": state,
                    "Δ기울기_ON": round(float(e["Δ기울기"].mean()), 2),
                    "Δ기울기_대조": round(float(ctl.mean()), 2),
                    "효과": round(float(effect), 2), "Cohen_d": round(float(d_coh), 2),
                    "임계값": round(thr, 1), "현재CUSUM": round(float(cs[-1]), 2)}


EXCH, CTL, rows = {}, {}, []
for tk in TANKS:
    EXCH[tk], CTL[tk], info = exch_model(tk)
    rows.append(info)
pd.DataFrame(rows)

## 5. 검증 ① — 오경보율

**정상 데이터에서 알람이 몇 번 뜨는가.** 이 4.7일 구간은 일별 중앙값이 거의 고정
(P1 0.123~0.127, I3 1.367~1.401)이라 실제 고장이 없었다고 본다. 그러니
여기서 뜨는 알람은 전부 오경보다. **0 이 나와야 쓸 수 있다.**

In [ ]:
rows = []
for tk in TANKS:
    r = {"탱크": tk}
    for name, src, col in [("feeding", FEED, "밸브유량"), ("exchanger", EXCH, "Δ기울기")]:
        e = src[tk]
        if "cusum" not in e.columns or len(e) < BASELINE_WINDOW + 20:
            r[f"{name}_오경보"] = None
            r[f"{name}_관측"] = len(e)
            continue
        thr, _, cs = calibrate(e[col].values.astype(float))
        # 임계값을 정상 최대값에서 뽑았으므로 정의상 0 이어야 한다.
        # 대신 **고정 임계값 5** 를 썼다면 몇 번 울렸을지도 같이 본다 (v1 의 실패 재현)
        r[f"{name}_관측"] = len(e)
        r[f"{name}_오경보"] = int((cs >= thr).sum())
        r[f"{name}_고정5였다면"] = int((cs >= 5.0).sum())
    rows.append(r)
pd.DataFrame(rows)

## 6. 검증 ② — 검출 곡선

**반응이 얼마나 약해지면, 몇 번째 이벤트에서 알람이 뜨는가.**
feeding 은 밸브유량을 그 비율만큼 깎고, exchanger 는 Δ기울기를 그 비율만큼 0 에
가깝게 만든다. 100% = 완전 정지.

실제 쓸 때 중요한 건 **"완전히 죽었을 때 빨리 잡느냐"** 와
**"절반만 약해졌을 때도 잡느냐"** 두 가지다.

In [ ]:
def detect_after(e, col, severity, n=20):
    if "cusum" not in e.columns or len(e) < BASELINE_WINDOW + 20:
        return None
    v = e[col].values.astype(float)
    thr, sign, _ = calibrate(v)
    vi = v.copy()
    vi[-n:] = vi[-n:] * (1 - severity)          # 반응이 severity 만큼 약해짐
    cs = cusum_down(rolling_z(vi * sign))[-n:]
    return int(np.argmax(cs >= thr)) + 1 if (cs >= thr).any() else None


rows = []
for tk in TANKS:
    r = {"탱크": tk}
    for name, src, col in [("feed", FEED, "밸브유량"), ("exch", EXCH, "Δ기울기")]:
        for sev in (0.25, 0.5, 1.0):
            r[f"{name}_{int(sev*100)}%"] = detect_after(src[tk], col, sev)
    rows.append(r)
print("몇 번째 이벤트에서 확정 알람 (None = 20개 안에 못 잡음)")
pd.DataFrame(rows)

## 7. 검증 ③ — 단순 규칙 대비 얼마나 나은가

모델이 실제로 값을 더하는지 보려면 **아무 생각 없는 규칙**과 비교해야 한다.

- **단순 규칙**: 전체 중앙값의 50% 미만이면 알람 (이동 기준선도, CUSUM 도 없음)
- **모델**: 이동 기준선 + 로버스트 z + CUSUM

같은 정상 데이터에서 오경보 수를, 같은 고장 주입에서 검출 속도를 비교한다.

In [ ]:
rows = []
for tk in TANKS:
    for name, src, col in [("feeding", FEED, "밸브유량"), ("exchanger", EXCH, "Δ기울기")]:
        e = src[tk]
        if "cusum" not in e.columns or len(e) < BASELINE_WINDOW + 20:
            continue
        v = e[col].values.astype(float)
        med = np.median(v)

        naive_fa = int((np.abs(v) < np.abs(med) * 0.5).sum())     # 정상 데이터 오경보
        vi = v.copy(); vi[-20:] = 0.0
        naive_hit = np.argmax(np.abs(vi[-20:]) < np.abs(med) * 0.5) + 1

        thr, sign, cs = calibrate(v)
        model_fa = int((cs >= thr).sum())
        model_hit = detect_after(e, col, 1.0)

        rows.append({"탱크": tk, "요소": name,
                     "단순_오경보": naive_fa, "모델_오경보": model_fa,
                     "단순_검출": int(naive_hit), "모델_검출": model_hit})
pd.DataFrame(rows)

## 8. 종합 — 탱크별로 실제 쓸 수 있는가

판정 기준 세 가지를 모두 만족해야 **사용 가능**이다.

1. **이벤트 수 ≥ 50** — 이동 기준선(30개)을 만들고도 감시할 여유가 있어야 한다
2. **오경보 0**
3. **완전 정지(100%)를 20 이벤트 안에 검출**

추가로 exchanger 는 **효과 Cohen's d ≥ 0.5** 를 요구한다. 그 미만이면 대조군과
구분 자체가 안 돼 알람을 믿을 수 없다.

In [ ]:
rows = []
for tk in TANKS:
    r = {"탱크": tk}
    # feeding
    e = FEED[tk]
    ok_n = len(e) >= 50 and "cusum" in e.columns
    hit = detect_after(e, "밸브유량", 1.0) if ok_n else None
    r["feeding_이벤트"] = len(e)
    r["feeding_검출"] = hit
    r["feeding_판정"] = ("사용가능" if (ok_n and hit is not None)
                        else ("표본부족" if len(e) < 50 else "검출불가"))
    # exchanger
    e2 = EXCH[tk]
    info = [x for x in [None]]
    ok_n2 = len(e2) >= 50 and "cusum" in e2.columns
    hit2 = detect_after(e2, "Δ기울기", 1.0) if ok_n2 else None
    ctl = CTL[tk]
    if ok_n2 and len(ctl) > 50:
        pooled = np.sqrt((e2["Δ기울기"].var() * (len(e2) - 1) + ctl.var() * (len(ctl) - 1))
                         / (len(e2) + len(ctl) - 2))
        dcoh = abs((e2["Δ기울기"].mean() - ctl.mean()) / pooled)
    else:
        dcoh = np.nan
    note = WINDOWS[tk][2]
    r["exch_이벤트"] = len(e2)
    r["exch_d"] = round(dcoh, 2) if dcoh == dcoh else None
    r["exch_검출"] = hit2
    if not note.startswith("OK") and not note.startswith("주기"):
        r["exch_판정"] = "측정불가(창)"
    elif not ok_n2:
        r["exch_판정"] = "표본부족"
    elif dcoh != dcoh or dcoh < 0.5:
        r["exch_판정"] = "효과미약"
    elif hit2 is None:
        r["exch_판정"] = "검출불가"
    else:
        r["exch_판정"] = "사용가능"
    rows.append(r)
VERDICT = pd.DataFrame(rows)
VERDICT

## 9. 예측이 실제와 얼마나 맞는가 (눈으로)

왼쪽: feeding 밸브유량이 이벤트마다 얼마나 일정한지 (일정할수록 이상 감지가 쉽다).
오른쪽: exchanger Δ기울기가 대조군과 얼마나 갈라지는지 (겹칠수록 판별 불가).

In [ ]:
TANK = "P2"

fig, axes = plt.subplots(2, 2, figsize=(15, 8))
e, x, ctl = FEED[TANK], EXCH[TANK], CTL[TANK]

axes[0][0].plot(e["start"], e["순유량"], ".", ms=4, alpha=0.6, label="순유량(보정 전)")
axes[0][0].plot(e["start"], e["밸브유량"], ".", ms=4, alpha=0.8, label="밸브유량(보정 후)")
axes[0][0].axhline(e["밸브유량"].median(), color="#d62728", ls="--", lw=1)
axes[0][0].set_title(f"{TANK} feeding — 유출 보정 전후"); axes[0][0].legend(fontsize=8)

axes[1][0].plot(e["start"], e["cusum"], color="#2ca02c", lw=1.2)
thr, _, _ = calibrate(e["밸브유량"].values.astype(float))
axes[1][0].axhline(thr, color="red", ls="--", lw=1, label=f"임계 {thr:.1f}")
axes[1][0].set_title("feeding CUSUM"); axes[1][0].legend(fontsize=8)

bins = np.linspace(min(x["Δ기울기"].min(), ctl.min()), max(x["Δ기울기"].max(), ctl.max()), 60)
axes[0][1].hist(ctl, bins=bins, alpha=0.55, label="대조군(exch OFF)", color="#888888", density=True)
axes[0][1].hist(x["Δ기울기"], bins=bins, alpha=0.55, label="exchanger ON", color="#2ca02c", density=True)
axes[0][1].axvline(ctl.mean(), color="#555555", ls="--", lw=1)
axes[0][1].axvline(x["Δ기울기"].mean(), color="#1a7a1a", ls="--", lw=1)
axes[0][1].set_title(f"{TANK} exchanger — Δ기울기 분포"); axes[0][1].legend(fontsize=8)

axes[1][1].plot(x["start"], x["cusum"], color="#2ca02c", lw=1.2)
thr2, _, _ = calibrate(x["Δ기울기"].values.astype(float))
axes[1][1].axhline(thr2, color="red", ls="--", lw=1, label=f"임계 {thr2:.1f}")
axes[1][1].set_title("exchanger CUSUM"); axes[1][1].legend(fontsize=8)

for a in axes.ravel():
    a.grid(alpha=0.3)
fig.autofmt_xdate()
fig.tight_layout()